## T10

In [117]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as st
import scipy.optimize as opt
import seaborn as sns
import math as mt
from skimage.io import imread, imsave

np.random.seed(42)

In [118]:
alpha = 0.05
k = 10
n = 100
nums = np.arange(k)
m = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])
ons = np.array([1 for _ in range(k)])

def Kolm(x): #распределение Колмогорова
    ans = 1.0
    for i in range(1, 1001):
        ans += 2*(-1)**i*np.exp(-2*(i**2)*(x**2))
    return ans 

def L(thetas):
    global m
    loc = thetas[0]
    scale = thetas[1]
    pi = st.norm.cdf(nums[1:], loc=loc, scale=scale) # вычисляем p_i
    prod = 1
    for i in range(8):
        prod *= (pi[i + 1]-pi[i])**m[i+1]

    return - (pi[0]**m[0]*(1 - pi[-1])**m[-1]*prod) # функция правдоподобия
 

a) Проверить гипотезу о согласии данных с законом равномерного распределения R(0,10) при помощи критерия $\chi^2$ и критерия Колмогорова

In [119]:
#Критерий chi^2
nps = 10*ons
delt_pearson_a = float(np.sum(((m-nps)**2)/nps))
pval_pearson_a = 1 - float(st.chi2(k-1).cdf(delt_pearson_a)) # по сути интеграл от значения до бесконечности это 1 - функция
                                         # распределения в этой точке
print("Критерий chi^2:")
print(f"Статистика критерия (дельта с волной) = {delt_pearson_a}, p-value = {pval_pearson_a:.4f}")
print("Нет оснований отвергнуть H0" if pval_pearson_a>alpha else "Отвергаем H0")

Критерий chi^2:
Статистика критерия (дельта с волной) = 16.4, p-value = 0.0590
Нет оснований отвергнуть H0


In [120]:
#Критерий Колмогорова:
F_emp = np.array([np.sum(m[:i]) for i in range(len(m)+1)]) / 100
F_th = np.array([np.sum(ons[:i]) for i in range(len(ons))]) / 10

delt_kolmogorov_a = float(np.sqrt(n)*np.max([max(np.abs(F_emp[i] - F_th[i]), np.abs(F_emp[i+1] - F_th[i])) for i in range(k)]))
pval_kolmogorov_a = 1 - Kolm(delt_kolmogorov_a)
print("Критерий Колмогорова:")
print(f"Статистика критерия (дельта с волной) = {delt_kolmogorov_a:.2f}, p-value = {pval_kolmogorov_a:.4f}")
print("Нет оснований отвергнуть H0" if pval_kolmogorov_a>alpha else "Отвергаем H0")

Критерий Колмогорова:
Статистика критерия (дельта с волной) = 1.40, p-value = 0.0397
Отвергаем H0


В данном случае Критерий Колмогорова проявил себя лучше: он точнее отвергнул нулевую гипотезу, когда критерий $\chi^2$ не нашел веских причин чтобы отвергнуть ее 

b) Проверить гипотезу о согласии данных с законом равномерного распределения N($\theta_1$,$\theta_2^2$) при помощи критерия $\chi^2$ и критерия Колмогорова

In [121]:
# Критерий chi^2

res = opt.differential_evolution( # максимизируем численным методом
    func=L,
    bounds=[(0, 10), (0, 10)],
    maxiter=10000,
)

theta1 = float(res.x[0])
theta2 = float(res.x[1])
pi = st.norm.cdf(nums[1:], loc=theta1, scale=theta2)
nps = []
nps.append(n * pi[0])
for i in range(8):
    nps.append(n * (pi[i+1] - pi[i]))
nps.append(n * (1 - pi[-1]))
nps = np.array(nps)
delt_pearson_b = float(np.sum(((m - nps)**2)/nps))
pval_pearson_b = float(1 - st.chi2(k-2-1).cdf(delt_pearson_b)) #кол-во неизвестных параметров в модели - 2
print("Критерий chi^2:")
print(f"Статистика критерия (дельта с волной) = {delt_pearson_b}, p-value = {pval_pearson_b:.4f}")
print("Нет оснований отвергнуть H0" if pval_pearson_b > alpha else "Отвергаем H0")

Критерий chi^2:
Статистика критерия (дельта с волной) = 9.805516418224453, p-value = 0.1999
Нет оснований отвергнуть H0


In [125]:
#Критерий Колмогорова:
N_BS = 50000
true_sample = []
for i in range(k):
    true_sample += [i]*m[i]
true_sample = np.array(true_sample)
mean = np.mean(true_sample)
std = np.std(true_sample, ddof=1)

def norm(x, mn, std):
    return 0.5 * (1 + mt.erf((x - mn)/(np.sqrt(2) * std)))

delt_kolmogorov_b = np.max([np.sqrt(n) * max(np.abs(norm(nums[i], mean, std) - F_emp[i]),np.abs(norm(nums[i], mean, std) - F_emp[i+1])) for i in range(10)])

def bootstrap(N_BS):
    bs_arr=[]
    model = st.norm(loc = mean, scale = std)
    for _ in range(N_BS):
        sample = np.sort(model.rvs(size=n))
        bs_F_emp = [i/n for i in range(n+1)]
        bs_mean = np.mean(sample)
        bs_std = np.std(sample, ddof=1)
        bs_arr.append(np.max([np.sqrt(n) * max(np.abs(norm(sample[j], bs_mean, bs_std) - bs_F_emp[j]), \
                                        np.abs(norm(sample[j], bs_mean, bs_std) - bs_F_emp[j+1])) \
                                            for j in range(len(sample))]))
    return np.array(bs_arr)

bs = bootstrap(N_BS=N_BS)
pval_kolmogorov_b = len(bs[bs >= delt_kolmogorov_b]) / N_BS
print("Критерий Колмогорова:")
print(f"Статистика критерия (дельта с волной) = {delt_kolmogorov_b:.2f}, p-value = {pval_kolmogorov_b:.4f}")
print("Нет оснований отвергнуть H0" if pval_kolmogorov_b>alpha else "Отвергаем H0")

Критерий Колмогорова:
Статистика критерия (дельта с волной) = 1.00, p-value = 0.0151
Отвергаем H0


Аналогично пункту **a)** наблюдаем более точное поведение у критерия Колмогорова